# Binance daily SVM pilot: step-by-step walkthrough

This notebook exposes the complete approved BTCUSDT pilot one stage at a time. It reuses the repository modules that power `scripts/run_daily_svm_pilot.py`; the notebook does not define an alternative research pipeline.

The stages are: configuration, immutable raw data, internal-gap treatment, feature construction, next-day D0 targets, one transparent rolling training step, the complete rolling SVM pilot, and persisted outputs.

This walkthrough uses the shared daily stages and centralized artifact paths. Install the project with `python -m pip install -e ".[dev]"` before opening the notebook. Outputs were cleared after refactoring; execute cells in order to generate current evidence. `OVERWRITE` defaults to `False`. No final-test evaluator is executed here.


## Methodological boundaries

- Inputs at date `t` are mapped to the next day's outcome: `features(t) -> D0(t + 1)`.
- `D0(t + 1) = high(t + 1) / open(t + 1) - 1`.
- The independent binary thresholds are 2%, 3%, and 4%.
- Internal missing UTC days are forward-filled only in derived data and marked with `is_imputed`; raw files remain unchanged.
- Feature selection and `StandardScaler` are refitted on each training window only. The immediately following observation is transformed but never used for fitting.
- The pilot uses the default parameters of scikit-learn `SVC`. This notebook does not perform grid search, portfolio construction, or backtesting.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


from tcc_crypto.paths import repository_root
from tcc_crypto.workflows.context import load_context
from tcc_crypto.workflows import daily_data
from tcc_crypto.workflows.svm import persist_results
from tcc_crypto.features.catalog import ACTIVE_FEATURES, feature_definitions
from tcc_crypto.features.warmup import remove_leading_feature_warmup
from tcc_crypto.data.dataset import sample_features as select_sample_features
from tcc_crypto.modeling.preprocessing import prepare_window

REPOSITORY_ROOT = repository_root()
from tcc_crypto.data.loaders import load_daily_feature_input
from tcc_crypto.modeling.pilot import run_default_svc_rolling_pilot, fit_prediction

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository root: {REPOSITORY_ROOT}')

## 1. Load and inspect the experiment configuration

The JSON files hold the approved parameters; the Python modules validate and execute them. Keeping parameters outside the implementation makes each run explicit and reproducible.

In [ ]:
context = load_context('pilot', root=REPOSITORY_ROOT)
pilot_config = context.model
feature_config = context.features
OVERWRITE = False  # Set True explicitly to replace generated artifacts.
display(pd.Series(context.paths.inventory(context.kind), name='resolved_path').to_frame())
display(pd.DataFrame(feature_definitions()))

configuration_overview = pd.Series(
    {
        'dataset_id': pilot_config.dataset_id,
        'symbol': pilot_config.symbol,
        'warmup_start_utc': pilot_config.warmup_start_utc,
        'sample_start_utc': pilot_config.sample_start_utc,
        'sample_end_utc': pilot_config.sample_end_utc,
        'target_thresholds': pilot_config.target_thresholds,
        'training_window_sizes': pilot_config.training_window_sizes,
        'estimator': pilot_config.estimator,
        'estimator_parameters': pilot_config.estimator_parameters,
        'scaler': pilot_config.scaler,
        'feature_count': len(feature_config.active_features),
        'output_directory': pilot_config.output_directory,
    },
    name='configured_value',
)
display(configuration_overview.to_frame())

## 2. Load the immutable raw daily observations

This stage reads the previously collected Binance CSV. It does not contact Binance and does not modify the raw file. The preview verifies the timestamp range, schema, and OHLC values before any derived treatment.

In [ ]:
raw_path = daily_data.raw_source(context)
raw_values = load_daily_feature_input(raw_path, fill_internal_gaps=False)

raw_overview = pd.Series(
    {
        'source_file': raw_path,
        'row_count': len(raw_values),
        'first_timestamp_utc': raw_values.index.min(),
        'last_timestamp_utc': raw_values.index.max(),
        'duplicate_timestamps': int(raw_values.index.duplicated().sum()),
        'missing_cells': int(raw_values.isna().sum().sum()),
    },
    name='raw_data',
)
display(raw_overview.to_frame())
display(raw_values.head(5))

## 3. Apply the approved internal-gap treatment

`load_daily_feature_input(..., fill_internal_gaps=True)` calls the repository's gap-treatment layer. It creates only dates bounded by real observations, carries the preceding daily observation forward, and marks synthesized rows. An empty imputed-row preview means that this asset has no internal daily gaps; it is not an error.

In [ ]:
daily_data.align(context, overwrite=OVERWRITE)
aligned_values = daily_data.read_stage(context, 'aligned')
imputed_values = aligned_values.loc[aligned_values['is_imputed']].copy()

gap_overview = pd.Series(
    {
        'raw_rows': len(raw_values),
        'aligned_rows': len(aligned_values),
        'imputed_internal_days': int(aligned_values['is_imputed'].sum()),
        'raw_file_was_modified': False,
    },
    name='gap_treatment',
)
display(gap_overview.to_frame())
display(imputed_values.head(10))

## 4. Calculate features with warm-up history

June 2019 is loaded only to provide earlier observations required by lags and technical indicators. After feature calculation, only leading incomplete rows are removed. The effective pilot sample begins on July 1, 2019.

In [ ]:
daily_data.features(context, overwrite=OVERWRITE)
calculation_values = aligned_values
features_with_warmup = daily_data.read_stage(context, 'features')
complete_features = remove_leading_feature_warmup(features_with_warmup)
sample_features = select_sample_features(features_with_warmup, context.data)

feature_overview = pd.Series(
    {
        'calculation_rows_including_warmup': len(calculation_values),
        'leading_rows_removed_after_indicators': len(features_with_warmup) - len(complete_features),
        'pilot_feature_rows': len(sample_features),
        'first_pilot_feature_date': sample_features.index.min(),
        'last_pilot_feature_date': sample_features.index.max(),
        'feature_count': sample_features.shape[1],
        'missing_pilot_feature_cells': int(sample_features.isna().sum().sum()),
    },
    name='feature_construction',
)
display(feature_overview.to_frame())
display(sample_features.head(5))

## 5. Construct and inspect the next-day D0 targets

For every feature row at `t`, the target builder reads the open and high from `t + 1`. The return is converted independently into three labels. The displayed rows make this alignment auditable.

In [ ]:
daily_data.targets(context, overwrite=OVERWRITE)
targets = daily_data.read_stage(context, 'targets')
target_columns = ['target_2pct', 'target_3pct', 'target_4pct']
display(
    targets.loc[
        :,
        ['target_date_utc', 'next_day_open', 'next_day_high', 'next_day_d0', *target_columns],
    ].head(10)
)

In [ ]:
target_balance = pd.DataFrame(
    {
        'negative': [(targets[column] == 0).sum() for column in target_columns],
        'positive': [(targets[column] == 1).sum() for column in target_columns],
    },
    index=target_columns,
)
target_balance['positive_rate'] = (
    target_balance['positive'] / target_balance[['negative', 'positive']].sum(axis=1)
)
display(target_balance)

ax = target_balance[['negative', 'positive']].plot.bar(
    figsize=(8, 4),
    color=['#4472C4', '#ED7D31'],
)
ax.set_title('BTCUSDT target-class balance — 2019-07-01 to 2019-12-31')
ax.set_xlabel('Binary target')
ax.set_ylabel('Daily observations')
ax.legend(title='Class')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Assemble the official modeling dataset

The prior cells exposed the internals for inspection. This cell now calls `build_daily_modeling_dataset`, the same orchestration function used by the executable script, and verifies the resulting modeling table.

In [ ]:
daily_data.dataset(context, overwrite=OVERWRITE)
modeling_dataset = daily_data.read_stage(context, 'dataset')

dataset_overview = pd.Series(
    {
        'rows': len(modeling_dataset),
        'columns': modeling_dataset.shape[1],
        'feature_columns': len(ACTIVE_FEATURES),
        'feature_imputed_rows': int(modeling_dataset['feature_is_imputed'].sum()),
        'target_imputed_rows': int(modeling_dataset['target_is_imputed'].fillna(False).sum()),
        'first_date': modeling_dataset.index.min(),
        'last_date': modeling_dataset.index.max(),
    },
    name='modeling_dataset',
)
display(dataset_overview.to_frame())
display(modeling_dataset.head(5))

## 7. Inspect one rolling SVM training step

Before running every combination, this section reproduces one step transparently: the first 60 observations train the 2% target model, and observation 61 is the out-of-sample prediction row. This is an educational view of the same operations implemented by `run_default_svc_rolling_pilot`.

In [ ]:
example_target = 'target_2pct'
example_window_size = 60
feature_matrix = modeling_dataset.loc[:, ACTIVE_FEATURES].astype('float64')
labels = modeling_dataset[example_target].astype('int64')

training_features = feature_matrix.iloc[:example_window_size]
prediction_features = feature_matrix.iloc[example_window_size:example_window_size + 1]
training_labels = labels.iloc[:example_window_size]
actual_label = int(labels.iloc[example_window_size])

window_overview = pd.Series(
    {
        'target': example_target,
        'window_size': example_window_size,
        'training_start_utc': training_features.index[0],
        'training_end_utc': training_features.index[-1],
        'prediction_time_utc': prediction_features.index[0],
        'training_negative': int((training_labels == 0).sum()),
        'training_positive': int((training_labels == 1).sum()),
        'actual_out_of_sample_label': actual_label,
    },
    name='first_rolling_step',
)
display(window_overview.to_frame())

In [ ]:
prepared = prepare_window(training_features, prediction_features, feature_config.feature_selection)
selector = prepared.selector
selected_training = selector.transform(training_features)
selected_prediction = selector.transform(prediction_features)

selection_overview = pd.Series(
    {
        'input_feature_count': training_features.shape[1],
        'selected_feature_count': len(selector.selected_features_),
        'selected_features': ', '.join(selector.selected_features_),
        'fit_rows': len(selected_training),
        'transformed_future_rows': len(selected_prediction),
    },
    name='training-only feature selection',
)
display(selection_overview.to_frame())

In [ ]:
scaler = prepared.scaler
scaled_training = prepared.training
scaled_prediction = prepared.evaluation
scaled_training_frame = pd.DataFrame(
    scaled_training,
    index=selected_training.index,
    columns=selected_training.columns,
)

scaling_diagnostics = pd.DataFrame(
    {
        'training_mean_after_scaling': scaled_training_frame.mean(),
        'training_std_after_scaling': scaled_training_frame.std(ddof=0),
        'prediction_scaled_value': scaled_prediction[0],
    }
)
display(scaling_diagnostics.head(10))
print('The scaler was fitted on training rows only; the prediction row was only transformed.')

In [ ]:
example = fit_prediction(training_labels, scaled_training, scaled_prediction)
example_result = pd.Series({**example, 'actual': actual_label}, name='one out-of-sample prediction')
display(example_result.to_frame())


## 8. Run the complete rolling default-SVC pilot

The repository function repeats the demonstrated sequence for every prediction date, each target, and each 60-, 75-, and 90-day window. Rows whose training window contains only one class are recorded explicitly and do not receive an invented prediction.

In [ ]:
pilot_result = run_default_svc_rolling_pilot(
    modeling_dataset,
    selection_config=feature_config.feature_selection,
)

run_overview = pd.Series(
    {
        'rolling_result_rows': len(pilot_result.predictions),
        'successful_predictions': int((pilot_result.predictions['status'] == 'ok').sum()),
        'single_class_windows': int((pilot_result.predictions['status'] == 'single_class_training').sum()),
        'summary_rows': len(pilot_result.summary),
    },
    name='complete_pilot',
)
display(run_overview.to_frame())
display(pilot_result.predictions.head(10))

## 9. Inspect in-sample diagnostics and out-of-sample metrics

In-sample metrics describe the fitted rolling windows. The `out_of_sample_*` columns aggregate the one-step-ahead predictions and are the appropriate pilot view of behavior on unseen rows. These diagnostics validate the pipeline; they do not select final hyperparameters.

In [ ]:
display(pilot_result.summary)

metric_columns = [
    'out_of_sample_accuracy',
    'out_of_sample_precision',
    'out_of_sample_recall',
    'out_of_sample_f1',
]
metric_plot = pilot_result.summary.set_index(['target', 'window_size'])[metric_columns]
ax = metric_plot.plot.bar(figsize=(13, 5))
ax.set_title('Default SVC out-of-sample diagnostics — BTCUSDT 2019H2')
ax.set_xlabel('Target and rolling training-window size')
ax.set_ylabel('Metric value')
ax.set_ylim(0, 1)
ax.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Persist the same artifacts as the executable pipeline

The final step writes the derived modeling dataset, rolling predictions, metric summary, and reproducibility metadata to the configured output directory. These are generated artifacts and remain outside Git.

In [ ]:
persist_results(context, pilot_result, overwrite=OVERWRITE)
written_artifacts = pd.Series(context.paths.inventory(context.kind), name='path').to_frame()
display(written_artifacts)


## Interpretation limits and next step

This notebook demonstrates the approved default-SVC pilot and its temporal safeguards. It does not establish the final classifier. The subsequently approved temporal grid search is explained in `15_binance_daily_svm_grid_search_walkthrough.ipynb`. Neither notebook constructs portfolios or reports final-test performance.